<a href="https://colab.research.google.com/github/sathvikteja/MachineLearningAlgorithms/blob/main/CarPricePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving carprice.zip to carprice (1).zip


In [ ]:
!unzip carprice.zip

Archive:  carprice.zip
replace Used_Car_Price_Prediction.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os
os.listdir()

['.config',
 'Used_Car_Price_Prediction.csv',
 'carprice.zip',
 'carprice (1).zip',
 'sample_data']

In [ ]:
import pandas as pd
df=pd.read_csv('Used_Car_Price_Prediction.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7400 entries, 0 to 7399
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   car_name             7400 non-null   object 
 1   yr_mfr               7400 non-null   int64  
 2   fuel_type            7400 non-null   object 
 3   kms_run              7400 non-null   int64  
 4   sale_price           7400 non-null   int64  
 5   city                 7400 non-null   object 
 6   times_viewed         7400 non-null   int64  
 7   body_type            7297 non-null   object 
 8   transmission         6844 non-null   object 
 9   variant              7400 non-null   object 
 10  assured_buy          7400 non-null   bool   
 11  registered_city      7390 non-null   object 
 12  registered_state     7390 non-null   object 
 13  is_hot               7400 non-null   bool   
 14  rto                  7400 non-null   object 
 15  source               7274 non-null   o

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
# Basic preprocessing
# Handle categorical columns with Label Encoding
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
  df[col] = LabelEncoder().fit_transform(df[col].astype(str))

In [ ]:
# Fill missing values with median
df = df.fillna(df.median())

In [ ]:
# Define features and target
X = df.drop("sale_price", axis=1)
y = df["sale_price"]

In [ ]:
# Split into train/validation/test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5,random_state=42)

In [ ]:
# Standardise features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


In [ ]:
# Linear Regression (Baseline)
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_val)


In [ ]:
# Assuming y_val and y_pred are defined
print("Validation RMSE (Linear):", mean_squared_error(y_val, y_pred))
print("Validation R2 (Linear):", r2_score(y_val, y_pred))


Validation RMSE (Linear): 0.21107608155643476
Validation R2 (Linear): 0.9999999999965603


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

degrees = range(1, 6)  # try degree 1 to 5 (you can change this)
results = []

for d in degrees:
    # Transform features
    poly = PolynomialFeatures(degree=d, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_val_poly = poly.transform(X_val)

    # Fit model
    model = LinearRegression()
    model.fit(X_train_poly, y_train)

    # Predict
    y_pred = model.predict(X_val_poly)

    # Evaluate
    r2 = r2_score(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    results.append((d, r2, rmse))

# Display results
print("Degree | Validation R² | RMSE")
for d, r2, rmse in results:
    print(f"{d:^7} | {r2:.4f} | {rmse:.4f}")

# Optional: choose best degree
best = max(results, key=lambda x: x[1])
print(f"\n✅ Best degree based on R²: {best[0]} (R² = {best[1]:.4f}, RMSE = {best[2]:.4f})")


In [ ]:
print("Validation RMSE (Polynomial deg=2):", mean_squared_error(y_val, y_pred_poly))
print("Validation R2 (Polynomial deg=2):", r2_score(y_val, y_pred_poly))

Validation RMSE (Polynomial deg=2): 0.20441240610187564
Validation R2 (Polynomial deg=2): 0.9999999999966689


In [ ]:
# Ridge (L2 regularisation)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_val)
rmse_ridge = np.sqrt(mean_squared_error(y_val, y_pred_ridge))
print("Validation RMSE (Ridge):", rmse_ridge)

# Lasso (L1 regularisation)
lasso = Lasso(alpha=0.01)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_val)
rmse_lasso = np.sqrt(mean_squared_error(y_val, y_pred_lasso))
print("Validation RMSE (Lasso):", rmse_lasso)

In [ ]:
# If X_train is a DataFrame, convert to numpy array
X_train_np = np.array(X_train)

# Add bias column (intercept)
X_train_bias = np.c_[np.ones(X_train_np.shape[0]), X_train_np]  # shape: m x (n+1)

def gradient_descent(X, y, lr=0.01, epochs=1000):
    m, n = X.shape
    theta = np.zeros(n)
    for epoch in range(epochs):
        gradients = -(2/m) * X.T.dot(y - X.dot(theta))
        theta -= lr * gradients
    return theta

# Make sure y_train is also a numpy array
y_train_np = np.array(y_train)

# Apply gradient descent
theta = gradient_descent(X_train_bias, y_train_np, lr=0.01, epochs=1000)
print("Theta learned by GD (first 5):", theta[:5], "...")

In [ ]:
# Plot predictions vs actual values for Polynomial Regression
plt.scatter(y_val, y_pred_poly, alpha=0.5)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("Polynomial Regression (deg=2): Actual vs Predicted")
plt.show()

In [ ]:
# Plot residuals for Ridge Regression
residuals = y_val - y_pred_ridge
plt.hist(residuals, bins=30, edgecolor="k")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Ridge Regression Residuals")
plt.show()
